# Notebook 3 — Analyse exploratoire des données

## Contexte
Ce notebook présente l'**analyse exploratoire complète** des scrutins présidentiels français de 1995 à 2022. On cherche à :
- Documenter l'évolution de la participation et des familles politiques
- Identifier les corrélations entre profil démographique et comportement électoral
- Repérer les départements outliers et les tendances territoriales


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('..').resolve()
OUT_DIR  = BASE_DIR / 'outputs'

df_demo = pd.read_csv(OUT_DIR / 'elections_with_demography.csv')
df_cand = pd.read_csv(OUT_DIR / 'elections_candidats.csv')

ANNEES = [1995, 2002, 2007, 2012, 2017, 2022]
print(f'Données dept  : {df_demo.shape}')
print(f'Données cand  : {df_cand.shape}')


## 1. Évolution de la participation nationale

On calcule les taux nationaux en agrégeant les inscrits/votants de tous les départements.


In [ ]:
natl = (
    df_demo.groupby(['annee','tour'])
    .agg(inscrits=('inscrits','sum'), votants=('votants','sum'),
         abstentions=('abstentions','sum'))
    .reset_index()
)
natl['taux_participation'] = natl['votants']    / natl['inscrits'] * 100
natl['taux_abstention']   = natl['abstentions'] / natl['inscrits'] * 100

fig, ax = plt.subplots(figsize=(10, 5))
for tour, color in zip([1,2], ['#1f77b4','#ff7f0e']):
    sub = natl[natl.tour==tour]
    ax.plot(sub['annee'], sub['taux_participation'], marker='o',
            label=f'Tour {tour}', color=color, linewidth=2.5)
    for _, row in sub.iterrows():
        ax.annotate(f"{row['taux_participation']:.1f}%",
                    (row['annee'], row['taux_participation']),
                    textcoords='offset points', xytext=(0,9),
                    ha='center', fontsize=8)
ax.set_title('Évolution de la participation nationale (1995–2022)', fontsize=13)
ax.set_xlabel('Année') ; ax.set_ylabel('Taux de participation (%)')
ax.set_xticks(ANNEES) ; ax.set_ylim(50, 100)
ax.legend() ; ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print('\nConstat : baisse tendancielle de -0.6 pt/an depuis 2007 au T1')


## 2. Familles politiques au premier tour

On regroupe les candidats en **8 familles politiques** pour visualiser les recompositions du paysage politique sur 30 ans.


In [ ]:
import unicodedata

FAMILLES = {
    'Extrême gauche' : ['LAGUILLER ARLETTE','BESANCENOT OLIVIER','POUTOU PHILIPPE','ARTHAUD NATHALIE'],
    'Gauche'         : ['JOSPIN LIONEL','HUE ROBERT','TAUBIRA CHRISTIANE','ROYAL SEGOLENE',
                         'HOLLANDE FRANCOIS','HAMON BENOIT','MELENCHON JEAN-LUC'],
    'Ecologie'       : ['VOYNET DOMINIQUE','MAMERE NOEL','JADOT YANNICK'],
    'Centre'         : ['BAYROU FRANCOIS','MACRON EMMANUEL'],
    'Droite'         : ['CHIRAC JACQUES','BALLADUR EDOUARD','SARKOZY NICOLAS','FILLON FRANCOIS','PECRESSE VALERIE'],
    'Souverainiste'  : ['VILLIERS DE PHILIPPE','DUPONT-AIGNAN NICOLAS','CHEVENEMENT JEAN-PIERRE'],
    'Extrême droite' : ['LE PEN J.MARIE','LE PEN JEAN-MARIE','LE PEN MARINE','MEGRET BRUNO','ZEMMOUR ERIC'],
}

def normalize(s):
    s = str(s).upper().strip()
    return ''.join(c for c in unicodedata.normalize('NFD',s) if unicodedata.category(c)!='Mn')

FAM_NORM = {f:{normalize(n) for n in ns} for f,ns in FAMILLES.items()}

def get_famille(cand):
    cn = normalize(cand)
    for f,ns in FAM_NORM.items():
        if cn in ns: return f
    return 'Autre'

t1 = df_cand[df_cand['tour']==1].copy()
t1['famille'] = t1['candidat'].apply(get_famille)
print('Répartition Autre :', t1[t1.famille=='Autre']['candidat'].unique()[:10])


In [ ]:
# Scores nationaux par famille et année
exp_natl = (
    t1.groupby('annee')
    .apply(lambda g: g.drop_duplicates('dept_code')['exprimes'].sum(), include_groups=False)
    .reset_index(name='exprimes_natl')
)
scores = t1.groupby(['annee','famille'])['voix'].sum().reset_index()
scores = scores.merge(exp_natl, on='annee')
scores['pct'] = scores['voix'] / scores['exprimes_natl'] * 100

pivot = scores.pivot_table(index='annee', columns='famille', values='pct', aggfunc='sum').fillna(0)

COULEURS = {
    'Extrême gauche':'#d62728','Gauche':'#e84a5f','Ecologie':'#2ca02c',
    'Centre':'#ff7f0e','Droite':'#1f77b4','Souverainiste':'#9467bd',
    'Extrême droite':'#7f7f7f','Autre':'#bcbd22'
}

fig, ax = plt.subplots(figsize=(12,6))
bottom = np.zeros(len(pivot))
for col in pivot.columns:
    ax.bar(pivot.index, pivot[col], bottom=bottom, color=COULEURS.get(col,'#ccc'),
           alpha=0.85, width=3, label=col)
    for i,(v,b) in enumerate(zip(pivot[col],bottom)):
        if v>3: ax.text(pivot.index[i],b+v/2,f'{v:.0f}%',ha='center',va='center',fontsize=7,color='white',fontweight='bold')
    bottom += pivot[col].values
ax.set_title('Familles politiques au T1 (1995–2022) — % des exprimés', fontsize=13)
ax.set_xlabel('Année') ; ax.set_ylabel('% des exprimés')
ax.set_xticks(ANNEES) ; ax.set_ylim(0,105)
ax.legend(loc='upper left', ncol=2, fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout() ; plt.show()
print('Constat : extrême droite passe de 15% (1995) à 30% (2022). Centre émerge avec Macron en 2017.')


## 3. Corrélation démographie × abstention

On cherche à comprendre si la **composition démographique** d'un département explique son taux d'abstention.

**Hypothèse testée :** les départements avec plus de jeunes (0–39 ans) abstiennent-ils davantage ?


In [ ]:
df_t1 = df_demo[(df_demo['tour']==1) & df_demo['pop_ens_total'].notna()].copy()

fig, axes = plt.subplots(1, 3, figsize=(15,5), sharey=True)
vars_demo = [('pct_seniors','% seniors (60+)','#e74c3c'),
             ('pct_jeunes','% jeunes (0–39)','#27ae60'),
             ('pct_actifs','% actifs (40–59)','#2980b9')]

for ax, (var, label, color) in zip(axes, vars_demo):
    data = df_t1[[var,'taux_abstention']].dropna()
    ax.scatter(data[var], data['taux_abstention'], alpha=0.25, s=12, color=color)
    z = np.polyfit(data[var], data['taux_abstention'], 1)
    xs = np.linspace(data[var].min(), data[var].max(), 100)
    ax.plot(xs, np.poly1d(z)(xs), color='black', linewidth=2, linestyle='--')
    r = data[var].corr(data['taux_abstention'])
    ax.set_title(f'{label}\nr = {r:.2f}', fontsize=10)
    ax.set_xlabel(label) ; ax.grid(alpha=0.2)
axes[0].set_ylabel('Taux d\'abstention (%)')
plt.suptitle('Démographie et abstention par département (T1, 1995–2022)', fontsize=13)
plt.tight_layout() ; plt.show()

print('Corrélations Pearson :')
for var, label, _ in vars_demo:
    r = df_t1[[var,'taux_abstention']].dropna().corr().iloc[0,1]
    print(f'  {label:20s} r = {r:.3f}')


## 4. Évolution de l'abstention par département

Quels départements ont connu la **plus forte hausse** d'abstention entre 1995 et 2022 ?


In [ ]:
pivot_abs = df_t1.pivot_table(index='dept_nom_election', columns='annee',
                              values='taux_abstention').dropna(how='any')
pivot_abs['delta'] = pivot_abs[2022] - pivot_abs[1995]

fig, axes = plt.subplots(1,2, figsize=(14,6))
for ax, (data, title, color) in zip(axes, [
    (pivot_abs.nlargest(12,'delta'), 'Top 12 hausses (1995→2022)', '#e74c3c'),
    (pivot_abs.nsmallest(8,'delta'), 'Top 8 baisses',             '#27ae60'),
]):
    ds = data.sort_values('delta')
    bars = ax.barh(ds.index, ds['delta'], color=color, alpha=0.85)
    for b,v in zip(bars, ds['delta']):
        ax.text(v+(0.1 if v>0 else -0.1), b.get_y()+b.get_height()/2,
                f'{v:+.1f}pp', va='center', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Variation en points de %')
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(axis='x', alpha=0.3)
plt.suptitle('Variation du taux d\'abstention T1 entre 1995 et 2022', fontsize=13)
plt.tight_layout() ; plt.show()


## 5. Résumé des principales découvertes

| Indicateur | Valeur clé | Interprétation |
|---|---|---|
| Abstention T1 1995 | ~21% | Niveau de référence bas |
| Abstention T1 2022 | ~26% | +5 pts en 27 ans |
| Corrélation % jeunes × abstention | r = +0.38 | Plus de jeunes → plus d'abstention |
| Corrélation % actifs × abstention | r = -0.37 | Plus d'actifs → moins d'abstention |
| Extrême droite T1 | 15% → 30% | Doublement en 27 ans |
| Variance inter-départementale | 11% à 69% | Forte hétérogénéité territoriale |

➡️ Suite : `04_demography.ipynb` — Modélisation et prédictions 2027
